# 03 — Supervised Learning & Evaluation

**Primary:** Zhilin Zhang  
**Support:** Tuan Wei

Mandatory models: **K-Nearest Neighbours (KNN)** and **Decision Tree**.

The RQ is tested with two matched feature sets:
1. **size_only** — property-size attributes only;
2. **size_location_amenities** — the same size attributes plus location and amenity features.

The exact train/test split and high-price target are loaded from preprocessing so every section uses the same rows and threshold.

## 1. Imports and data

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_preprocessing.ipynb first.")

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target by split:")
display(pd.crosstab(df["split"], df["high_price"], margins=True))

## 2. Feature sets

In [ ]:
SIZE_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
]

LOCATION_FEATURES = [
    "distance_cbd_km",
]

AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]

FEATURE_SETS = {
    "size_only": SIZE_FEATURES,
    "size_location_amenities": (
        SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES
    ),
}

TARGET = "high_price"

for name, features in FEATURE_SETS.items():
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

train_df = df.loc[df["split"].eq("train")].copy()
test_df = df.loc[df["split"].eq("test")].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train positive rate:", round(train_df[TARGET].mean(), 4))
print("Test positive rate:", round(test_df[TARGET].mean(), 4))

## 3. Evaluation helpers

In [ ]:
def metric_summary(y_true, y_pred, y_score=None):
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    result = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "precision_class_0": float(precision[0]),
        "recall_class_0": float(recall[0]),
        "f1_class_0": float(f1[0]),
        "support_class_0": int(support[0]),
        "precision_class_1": float(precision[1]),
        "recall_class_1": float(recall[1]),
        "f1_class_1": float(f1[1]),
        "support_class_1": int(support[1]),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, y_score))
    else:
        result["roc_auc"] = np.nan

    return result


def bootstrap_macro_f1_ci(
    y_true,
    y_pred,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores.append(
            f1_score(
                y_true[idx],
                y_pred[idx],
                average="macro",
                zero_division=0,
            )
        )

    scores = np.asarray(scores)
    lo, hi = np.quantile(scores, [alpha/2, 1-alpha/2])

    return {
        "bootstrap_mean_macro_f1": float(scores.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "n_boot": int(n_boot),
    }

## 4. Majority-class baseline

The baseline is evaluated on the held-out test set with the same metrics as the trained models.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(
    np.zeros((len(train_df), 1)),
    train_df[TARGET],
)
baseline_pred = baseline.predict(np.zeros((len(test_df), 1)))
baseline_score = baseline.predict_proba(np.zeros((len(test_df), 1)))[:, 1]

baseline_metrics = metric_summary(
    test_df[TARGET].to_numpy(),
    baseline_pred,
    baseline_score,
)
baseline_metrics

## 5. Model pipelines and tuning grids

KNN uses median imputation + standardisation because it is distance-based.

Decision Tree uses median imputation but no scaling.

Both use 5-fold stratified CV on the training set and tune macro-F1. Every tried parameter combination is saved.

In [ ]:
def make_knn_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ])

def make_tree_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])

KNN_GRID = {
    "model__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

TREE_GRID = {
    "model__max_depth": [None, 3, 5, 8, 12],
    "model__min_samples_split": [2, 10, 30],
    "model__min_samples_leaf": [1, 5, 15, 30],
}

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("KNN combinations:", np.prod([len(v) for v in KNN_GRID.values()]))
print("Tree combinations:", np.prod([len(v) for v in TREE_GRID.values()]))

## 6. Tune and evaluate each model × feature-set combination

In [ ]:
def run_experiment(model_name, feature_set_name):
    features = FEATURE_SETS[feature_set_name]

    X_train = train_df[features]
    y_train = train_df[TARGET]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    if model_name == "KNN":
        pipeline = make_knn_pipeline()
        grid = KNN_GRID
    elif model_name == "DecisionTree":
        pipeline = make_tree_pipeline()
        grid = TREE_GRID
    else:
        raise ValueError(model_name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        scoring="f1_macro",
        cv=CV,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    pred = search.predict(X_test)
    proba = search.predict_proba(X_test)[:, 1]

    metrics = metric_summary(y_test.to_numpy(), pred, proba)
    metrics.update({
        "model": model_name,
        "feature_set": feature_set_name,
        "best_cv_macro_f1": float(search.best_score_),
        "best_params": json.dumps(search.best_params_, sort_keys=True),
        "absolute_macro_f1_improvement_vs_baseline": (
            metrics["macro_f1"] - baseline_metrics["macro_f1"]
        ),
    })

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.insert(0, "model", model_name)
    cv_results.insert(1, "feature_set", feature_set_name)

    return {
        "search": search,
        "pred": pred,
        "proba": proba,
        "metrics": metrics,
        "cv_results": cv_results,
        "confusion_matrix": confusion_matrix(y_test, pred, labels=[0, 1]),
        "features": features,
    }

experiments = {}

for model_name in ["KNN", "DecisionTree"]:
    for feature_set_name in FEATURE_SETS:
        print("Running:", model_name, feature_set_name)
        experiments[(model_name, feature_set_name)] = run_experiment(
            model_name,
            feature_set_name,
        )

model_summary = pd.DataFrame(
    [result["metrics"] for result in experiments.values()]
)

display(
    model_summary[
        [
            "model",
            "feature_set",
            "best_cv_macro_f1",
            "accuracy",
            "macro_f1",
            "roc_auc",
            "f1_class_0",
            "f1_class_1",
            "absolute_macro_f1_improvement_vs_baseline",
            "best_params",
        ]
    ].sort_values(["model", "feature_set"])
)

## 7. Incremental value of location + amenities

This table directly answers the first part of the RQ for each mandatory model by subtracting size-only performance from full-feature performance.

In [ ]:
incremental_rows = []

for model_name in ["KNN", "DecisionTree"]:
    size_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_only")
    ].iloc[0]
    full_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_location_amenities")
    ].iloc[0]

    incremental_rows.append({
        "model": model_name,
        "size_only_macro_f1": float(size_row["macro_f1"]),
        "full_macro_f1": float(full_row["macro_f1"]),
        "absolute_macro_f1_change": float(
            full_row["macro_f1"] - size_row["macro_f1"]
        ),
        "size_only_accuracy": float(size_row["accuracy"]),
        "full_accuracy": float(full_row["accuracy"]),
        "absolute_accuracy_change": float(
            full_row["accuracy"] - size_row["accuracy"]
        ),
        "size_only_roc_auc": float(size_row["roc_auc"]),
        "full_roc_auc": float(full_row["roc_auc"]),
        "absolute_roc_auc_change": float(
            full_row["roc_auc"] - size_row["roc_auc"]
        ),
    })

incremental_results = pd.DataFrame(incremental_rows)
display(incremental_results)

## 8. Uncertainty quantification

A bootstrap 95% confidence interval is calculated for held-out macro-F1 of the better full-feature model (chosen by test macro-F1). The group should discuss the limitation that model selection and interval estimation use the same held-out comparison when interpreting the result.

In [ ]:
full_results = {
    model: experiments[(model, "size_location_amenities")]
    for model in ["KNN", "DecisionTree"]
}

uncertainty_model = max(
    full_results,
    key=lambda m: full_results[m]["metrics"]["macro_f1"],
)
uncertainty_exp = full_results[uncertainty_model]

uncertainty = bootstrap_macro_f1_ci(
    test_df[TARGET].to_numpy(),
    uncertainty_exp["pred"],
    n_boot=2000,
    random_state=RANDOM_STATE,
)
uncertainty["model"] = uncertainty_model
uncertainty["feature_set"] = "size_location_amenities"

uncertainty

## 9. Feature influence for both models

Permutation importance is calculated on the same held-out test set for the two full-feature models, using macro-F1. This gives feature-level influence in the original feature space for both KNN and Decision Tree.

In [ ]:
importance_rows = []

for model_name in ["KNN", "DecisionTree"]:
    exp = experiments[(model_name, "size_location_amenities")]
    features = exp["features"]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    perm = permutation_importance(
        exp["search"].best_estimator_,
        X_test,
        y_test,
        scoring="f1_macro",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    for feature, mean_imp, std_imp in zip(
        features,
        perm.importances_mean,
        perm.importances_std,
    ):
        importance_rows.append({
            "model": model_name,
            "feature": feature,
            "permutation_importance_mean": float(mean_imp),
            "permutation_importance_std": float(std_imp),
        })

permutation_importance_table = pd.DataFrame(importance_rows)
display(
    permutation_importance_table.sort_values(
        ["model", "permutation_importance_mean"],
        ascending=[True, False],
    )
)

## 10. Save evaluation outputs and figures

In [ ]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

baseline_table = pd.DataFrame([{
    "model": "MajorityClassBaseline",
    "feature_set": "none",
    **baseline_metrics,
}])

baseline_table.to_csv(TABLE_OUT / "baseline_metrics.csv", index=False)
model_summary.to_csv(TABLE_OUT / "model_summary.csv", index=False)
incremental_results.to_csv(TABLE_OUT / "model_incremental_value.csv", index=False)
permutation_importance_table.to_csv(
    TABLE_OUT / "model_permutation_importance.csv",
    index=False,
)
pd.DataFrame([uncertainty]).to_csv(
    TABLE_OUT / "model_uncertainty.csv",
    index=False,
)

for (model_name, feature_set_name), exp in experiments.items():
    safe_model = model_name.lower()
    safe_set = feature_set_name.lower()

    keep_cols = [
        c for c in exp["cv_results"].columns
        if c.startswith("param_")
        or c in [
            "model",
            "feature_set",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "rank_test_score",
            "params",
        ]
    ]
    exp["cv_results"][keep_cols].to_csv(
        TABLE_OUT / f"cv_{safe_model}_{safe_set}.csv",
        index=False,
    )

    cm = pd.DataFrame(
        exp["confusion_matrix"],
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
    cm.to_csv(
        TABLE_OUT / f"confusion_{safe_model}_{safe_set}.csv"
    )

plot_df = model_summary.pivot(
    index="model",
    columns="feature_set",
    values="macro_f1",
)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(plot_df.index))
width = 0.36

ax.bar(
    x - width/2,
    plot_df["size_only"],
    width,
    label="Size only",
)
ax.bar(
    x + width/2,
    plot_df["size_location_amenities"],
    width,
    label="Size + location + amenities",
)
ax.axhline(
    baseline_metrics["macro_f1"],
    linestyle="--",
    linewidth=1.2,
    label="Majority baseline",
)
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("Held-out macro-F1")
ax.set_title("Model performance by feature set")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_OUT / "model_macro_f1_comparison.png", dpi=200)
plt.close(fig)

for model_name in ["KNN", "DecisionTree"]:
    imp = permutation_importance_table[
        permutation_importance_table["model"] == model_name
    ].sort_values("permutation_importance_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(
        imp["feature"],
        imp["permutation_importance_mean"],
        xerr=imp["permutation_importance_std"],
    )
    ax.set_xlabel("Permutation importance (macro-F1 decrease)")
    ax.set_title(f"{model_name} feature influence")
    fig.tight_layout()
    fig.savefig(
        FIG_OUT / f"permutation_importance_{model_name.lower()}.png",
        dpi=200,
    )
    plt.close(fig)

print("Saved model outputs to:", TABLE_OUT)
print("Saved model figures to:", FIG_OUT)

## 11. Hyperparameter evidence checklist

For the group-written Methodology/Discussion, use the saved CV tables to state:
- every value tried;
- each tuned parameter's default;
- selected value;
- the observed validation-score effect of changing it.

Also compare model behaviour/limitations and feature influence using the actual output tables rather than generic model descriptions.